# RQ3: Perceived Diversity Differences Between Conditions

Offboarding sample: participants who also completed the offboarding survey.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pingouin as pg

FIG_DIR = "../tables_figures/"

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 11,
    "axes.titlesize": 11,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "mathtext.fontset": "custom",
    "mathtext.rm": "Arial",
    "mathtext.it": "Arial:italic",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

ARM_LABELS = {"K": "Control", "M": "Treatment"}


def describe_by_arm(df, value_col):
    """n, mean, SD, skewness and excess kurtosis of value_col per condition."""
    return (
        df.groupby("arm")[value_col]
        .agg(
            n="count",
            mean="mean",
            sd="std",
            skew="skew",
            kurtosis=lambda x: x.kurt(),
        )
        .rename(index=ARM_LABELS)
        .round(2)
    )


activity = pd.read_csv("../data/offboarding_sample_round1.csv", sep=";", decimal=",", encoding="utf-8-sig")
print(activity.shape)
# Keep only the days a participant could have been active: from their first day (where their
# rows start) up to and including the day they received the offboarding survey
activity = activity[activity["day"] <= activity["offboarding_received_date"]]
print(activity.shape)

In [ ]:
diversity_vars = ["topics", "parties"]

# Survey answers repeat on every day row, so take one value per participant
diversity = activity.groupby("handle")[["arm"] + diversity_vars].first().reset_index()
diversity.shape

## Distributions

In [ ]:
ITEM_LABELS = {
    "topics": "Diversity of Topics",
    "parties": "Diversity of Parties",
}
bins = np.arange(0.5, 6, 1)  # one bin per scale point, 1 to 5

for var, xlabel in ITEM_LABELS.items():
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
    for ax, (arm, label) in zip(axes, ARM_LABELS.items()):
        subset = diversity.loc[diversity["arm"] == arm, var]
        ax.hist(subset, bins=bins, edgecolor="black")
        ax.set_xticks(range(1, 6))
        ax.set_title(label)
        ax.set_xlabel(xlabel)
        stats_text = (
            f"$n$ = {len(subset)}\n"
            f"$M$ = {subset.mean():.2f}\n"
            f"$SD$ = {subset.std():.2f}\n"
            f"Skewness = {subset.skew():.2f}\n"
            f"Kurtosis = {subset.kurt():.2f}"
        )
        ax.text(0.03, 0.97, stats_text, transform=ax.transAxes,
                ha="left", va="top", fontsize=10)
    axes[0].set_ylabel("Number of Participants")
    fig.tight_layout()
    fig.savefig(FIG_DIR + f"{var}_by_condition.png")
    plt.show()

In [ ]:
for var in diversity_vars:
    print(var)
    display(describe_by_arm(diversity, var))

In [ ]:
# All participants together
diversity[diversity_vars].agg(["mean", "std"]).round(2)

## T-Tests for Condition Differences

In [ ]:
for var in diversity_vars:
    print(var)
    display(pg.normality(diversity, dv=var, group="arm"))
    display(pg.homoscedasticity(diversity, dv=var, group="arm"))

In [ ]:
for var in diversity_vars:
    control = diversity.loc[diversity["arm"] == "K", var]
    treatment = diversity.loc[diversity["arm"] == "M", var]
    print(var)
    display(pg.ttest(treatment, control, correction=True))